# Comparaison des modèles : coûts, paramètres, énergie
Notebook focalisé sur la comparaison des modèles frontier : coût d'entraînement, taille (paramètres), compute, énergie estimée, ratios par paramètre/token.


**Plan**
- Charger `frontier_ai_models.csv` et parser les champs numériques.
- Comparer coûts et compute (scatter log-log, top modèles).
- Comparer coûts et paramètres, coût par milliard de paramètres.
- Estimer l'énergie par modèle (lignes avec puissance + durée) et la comparer au coût.
- Ratios d'efficacité : coût par 1e23 FLOP, coût par paramètre, énergie par token, compute par paramètre.
- Noter la donnée manquante sur la performance (métriques absentes dans ce CSV).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA_DIR = Path("..") / "data" / "ai_models"


In [ ]:
def parse_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    s = str(value).lower()
    s = s.replace(",", "").replace(" ", "").replace("~", "").replace("≈", "")
    s = s.replace("×10^", "e").replace("x10^", "e").replace("×10", "e").replace("x10", "e")
    s = s.replace("^", "e").replace(">", "").replace("<", "")
    m = re.match(r"([0-9.+\-e]+)([kmbt]?)", s)
    if not m:
        return np.nan
    num, suf = m.groups()
    try:
        base = float(num)
    except ValueError:
        return np.nan
    mult = {"k": 1e3, "m": 1e6, "b": 1e9, "t": 1e12}.get(suf, 1)
    return base * mult


In [ ]:
raw_df = pd.read_csv(DATA_DIR / "frontier_ai_models.csv")
df = raw_df.copy()
df["publication_year"] = pd.to_datetime(df["Publication date"], errors="coerce").dt.year

numeric_map = {
    "Training compute (FLOP)": "compute_flop",
    "Parameters": "parameters",
    "Training dataset size (gradients)": "train_tokens",
    "Training compute cost (2023 USD)": "train_cost_usd",
    "Training power draw (W)": "power_w",
    "Training time (hours)": "train_hours",
}

for src, tgt in numeric_map.items():
    df[tgt] = df[src].apply(parse_number)

coverage = df[["publication_year", *numeric_map.values()]].notna().sum().to_frame("non_null")
coverage


La performance (qualité modèle) n'est pas fournie dans ce CSV ; les comparaisons se basent donc sur compute, coûts, paramètres et énergie estimée.


In [ ]:
# Scatter coût vs compute
cost_df = df[["compute_flop", "train_cost_usd", "publication_year", "Model", "Organization"]].dropna()

fig, ax = plt.subplots()
sc = ax.scatter(cost_df["compute_flop"], cost_df["train_cost_usd"], c=cost_df["publication_year"], cmap="plasma", alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Compute (FLOP)")
ax.set_ylabel("Coût d'entraînement (USD 2023)")
ax.set_title("Coût vs compute")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

cost_top = cost_df.sort_values("train_cost_usd", ascending=False).head(10)
cost_top[["Model", "Organization", "publication_year", "compute_flop", "train_cost_usd"]]


Les modèles les plus chers se situent dans la zone 1e24-1e26 FLOP et dépassent souvent les centaines de millions USD.


In [ ]:
# Paramètres vs coût, et coût par milliard de paramètres
param_df = df[["parameters", "train_cost_usd", "Model", "Organization", "publication_year"]].dropna()
param_df["cost_per_billion_params"] = param_df["train_cost_usd"] / (param_df["parameters"] / 1e9)

fig, ax = plt.subplots()
sc = ax.scatter(param_df["parameters"], param_df["train_cost_usd"], c=param_df["publication_year"], cmap="viridis", alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Paramètres")
ax.set_ylabel("Coût (USD 2023)")
ax.set_title("Coût vs paramètres")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

cost_param_top = param_df.sort_values("cost_per_billion_params", ascending=False).head(10)
cost_param_top[["Model", "Organization", "publication_year", "parameters", "train_cost_usd", "cost_per_billion_params"]]


Le coût par milliard de paramètres met en évidence les modèles où l'efficacité économique par paramètre est faible (outliers à surveiller).


In [ ]:
# Coût unitaire par 1e23 FLOP
cost_df["cost_per_1e23"] = cost_df["train_cost_usd"] / (cost_df["compute_flop"] / 1e23)

fig, ax = plt.subplots()
ax.scatter(cost_df["publication_year"], cost_df["cost_per_1e23"], alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("Année")
ax.set_ylabel("Coût par 1e23 FLOP (USD)")
ax.set_title("Tendance du coût unitaire de compute")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

cost_df[["cost_per_1e23", "train_cost_usd"]].describe()


Le coût unitaire reste dispersé; pas de chute nette qui compenserait la hausse du compute.


In [ ]:
# Energie estimée et comparaison coût/énergie
energy_df = df[["Model", "Organization", "publication_year", "power_w", "train_hours", "train_tokens", "train_cost_usd"]].dropna()
energy_df["energy_mwh"] = energy_df["power_w"] * energy_df["train_hours"] / 1e6
energy_df["kwh_per_token"] = energy_df["energy_mwh"] * 1000 / energy_df["train_tokens"]

fig, ax = plt.subplots()
sc = ax.scatter(energy_df["energy_mwh"], energy_df["train_cost_usd"], c=energy_df["publication_year"], cmap="cividis", alpha=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energie (MWh)")
ax.set_ylabel("Coût (USD 2023)")
ax.set_title("Coût vs énergie estimée")
plt.colorbar(sc, ax=ax, label="Année")
ax.grid(True, which="both", linestyle=":", linewidth=0.7)
plt.tight_layout()
plt.show()

energy_top = energy_df.sort_values("energy_mwh", ascending=False).head(10)[["Model", "Organization", "publication_year", "energy_mwh", "kwh_per_token", "train_cost_usd"]]
energy_stats = energy_df[["energy_mwh", "kwh_per_token"]].describe()
energy_top, energy_stats


Les cas extrêmes montrent plusieurs milliers à dizaines de milliers de MWh par entraînement. Le ratio kWh/token varie fortement selon l'efficacité.


In [ ]:
# Ratios d'efficacité supplémentaires
ratios = df[["Model", "Organization", "publication_year", "parameters", "compute_flop", "train_tokens"]].dropna()
ratios["compute_per_param"] = ratios["compute_flop"] / ratios["parameters"]
ratios["compute_per_token"] = ratios["compute_flop"] / ratios["train_tokens"]

ratios_top = ratios.sort_values("compute_per_param", ascending=False).head(10)[["Model", "Organization", "publication_year", "parameters", "compute_flop", "compute_per_param", "compute_per_token"]]

ratios_stats = ratios[["compute_per_param", "compute_per_token"]].describe()
ratios_top, ratios_stats


Les ratios compute/param et compute/token aident à repérer les modèles particulièrement intensifs : utile pour les arguments d'efficience ou de contrainte énergétique.


## Conclusion
- Les modèles frontier montrent des coûts d'entraînement qui grimpent avec le compute et la taille modèle; le coût unitaire n'a pas baissé assez vite pour compenser la montée en échelle.
- L'énergie estimée (quand renseignée) atteint plusieurs milliers de MWh pour certains runs, avec une intensité par token très variable selon l'efficacité et le volume de données.
- La performance (qualité modèle) n'est pas présente dans ce CSV : à compléter via benchmarks publics ou citations si une comparaison de performance est requise.
- Les tableaux top (coût, coût/param, énergie) servent de shortlist pour la présentation et pour illustrer les contraintes économiques et énergétiques.
